# 00. Two-Stage MF Pipeline (ALS → CatBoost → Pairwise+HardNeg)

Единый, последовательный ноутбук:
1. Импорты и загрузка данных
2. Тюнинг ALS по Recall@10
3. Two-stage baseline (MF+popularity → CatBoost)
4. Pairwise + hard negatives
5. CatBoost baseline vs hardneg + сравнение
6. Финальный тюнинг CatBoost (на hardneg)

Важно: anti-leakage split внутри train.

## 1. Импорты и конфигурация

In [1]:
import os
import ast
import json
import time
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# CPU threads
N_CPU_THREADS = min(36, (os.cpu_count() or 8))
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(N_CPU_THREADS)

try:
    from implicit.als import AlternatingLeastSquares
except Exception as e:
    raise ImportError('Не найден implicit. Установите: pip install implicit') from e

try:
    from catboost import CatBoostRanker, Pool
    from catboost import utils as cb_utils
except Exception as e:
    raise ImportError('Не найден catboost. Установите: pip install catboost') from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# GPU/CPU switch for CatBoost
USE_GPU = True
try:
    has_gpu = cb_utils.get_gpu_device_count() > 0
except Exception:
    has_gpu = False

if not has_gpu:
    USE_GPU = False

print('CatBoost device:', 'GPU' if USE_GPU else 'CPU')

project_root = Path.cwd()
if not (project_root / 'artifacts').exists() and (project_root.parent / 'artifacts').exists():
    project_root = project_root.parent

CONFIG_PATH = project_root / 'configs' / 'base.json'
artifacts_root_dir = 'artifacts'
if CONFIG_PATH.exists():
    with CONFIG_PATH.open('r', encoding='utf-8') as f:
        _cfg = json.load(f)
    artifacts_root_dir = (
        _cfg.get('paths', {}).get('artifacts_root_dir')
        or _cfg.get('data', {}).get('artifacts_root_dir')
        or 'artifacts'
    )

ARTIFACTS_DIR = project_root / artifacts_root_dir
PROCESSED_DIR = ARTIFACTS_DIR / 'processed'
SPLITS_DIR = ARTIFACTS_DIR / 'splits'
EVAL_DIR = SPLITS_DIR / 'eval'

MODEL_FAMILY = 'two_stage_mf_catboost'

EXP_DIR = ARTIFACTS_DIR / 'experiments' / 'mf_tuning'
CAND_DIR = ARTIFACTS_DIR / 'candidates' / MODEL_FAMILY
MODELS_DIR = ARTIFACTS_DIR / 'models' / MODEL_FAMILY
METRICS_DIR = ARTIFACTS_DIR / 'metrics' / MODEL_FAMILY
PRED_DIR = ARTIFACTS_DIR / 'predictions' / MODEL_FAMILY
OUT_PAIRWISE_DIR = ARTIFACTS_DIR / 'processed' / 'two_stage_features_mf'

for d in [EXP_DIR, CAND_DIR, MODELS_DIR, METRICS_DIR, PRED_DIR, OUT_PAIRWISE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

K_VALUES = [5, 10, 20, 50]
K_MAX = max(K_VALUES)
TARGET_K = 10

CatBoost device: CPU


## 2. Загрузка данных (один раз)

In [2]:
def load_any(paths):
    for p in paths:
        if p.exists():
            if p.suffix == '.csv':
                return pd.read_csv(p), p
            if p.suffix == '.parquet':
                return pd.read_parquet(p), p
            if p.suffix == '.json':
                with p.open('r', encoding='utf-8') as f:
                    return json.load(f), p
            if p.suffix == '.npz':
                return np.load(p, allow_pickle=True), p
    return None, None

artifacts_map = {
    'train_interactions_implicit': [SPLITS_DIR / 'train_interactions_implicit.csv'],
    'mf_train_implicit': [PROCESSED_DIR / 'mf_train_implicit.csv'],
    'user_mapping': [PROCESSED_DIR / 'user_mapping.csv'],
    'item_mapping': [PROCESSED_DIR / 'item_mapping.csv'],
    'seen_items': [EVAL_DIR / 'seen_items_by_user.csv'],
    'val_ground_truth': [EVAL_DIR / 'val_ground_truth.csv'],
    'test_ground_truth': [EVAL_DIR / 'test_ground_truth.csv'],
    'eval_users_val': [EVAL_DIR / 'eval_users_val.csv'],
    'eval_users_test': [EVAL_DIR / 'eval_users_test.csv'],
    'item_metadata': [PROCESSED_DIR / 'item_metadata_cleaned.csv'],
    'item_tag_stats': [PROCESSED_DIR / 'item_tag_stats.csv'],
    'split_metadata': [SPLITS_DIR / 'split_metadata.json'],
}

loaded = {}
for name, paths in artifacts_map.items():
    obj, path = load_any(paths)
    loaded[name] = {'obj': obj, 'path': path}

status = pd.DataFrame([
    {'artifact': k, 'found': v['path'] is not None, 'path': str(v['path']) if v['path'] else None}
    for k, v in loaded.items()
])
print('Статус артефактов:')
display(status)

train_full = loaded['train_interactions_implicit']['obj']
mf_train = loaded['mf_train_implicit']['obj']
user_map = loaded['user_mapping']['obj']
item_map = loaded['item_mapping']['obj']
seen_items = loaded['seen_items']['obj']
val_gt = loaded['val_ground_truth']['obj']
test_gt = loaded['test_ground_truth']['obj']
eval_users_val = loaded['eval_users_val']['obj']
eval_users_test = loaded['eval_users_test']['obj']
item_meta = loaded['item_metadata']['obj']
item_tag_stats = loaded['item_tag_stats']['obj']
split_meta = loaded['split_metadata']['obj']

critical = {
    'train_interactions_implicit': train_full,
    'mf_train_implicit': mf_train,
    'user_mapping': user_map,
    'item_mapping': item_map,
    'seen_items': seen_items,
    'val_ground_truth': val_gt,
    'test_ground_truth': test_gt,
    'eval_users_val': eval_users_val,
    'eval_users_test': eval_users_test,
}
missing = [k for k, v in critical.items() if v is None]
assert len(missing) == 0, f'Не найдены критичные артефакты: {missing}'

for df in [train_full, mf_train, user_map, item_map, seen_items, val_gt, test_gt, eval_users_val, eval_users_test]:
    if df is None:
        continue
    if 'userId' in df.columns:
        df['userId'] = pd.to_numeric(df['userId'], errors='coerce')
    if 'movieId' in df.columns:
        df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce')

for df in [train_full, mf_train, seen_items, val_gt, test_gt, eval_users_val, eval_users_test]:
    if df is None:
        continue
    if 'userId' in df.columns:
        df.dropna(subset=['userId'], inplace=True)
        df['userId'] = df['userId'].astype('int64')
    if 'movieId' in df.columns:
        df.dropna(subset=['movieId'], inplace=True)
        df['movieId'] = df['movieId'].astype('int64')

Статус артефактов:


,artifact,found,path
0,train_interactions_implicit,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
1,mf_train_implicit,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
2,user_mapping,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
3,item_mapping,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
4,seen_items,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
5,val_ground_truth,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
6,test_ground_truth,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
7,eval_users_val,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
8,eval_users_test,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...
9,item_metadata,True,/Users/maybeabakarov/Desktop/РекомендацииПроек...


## 3. Общие маппинги, матрицы, метрики

In [3]:
user_to_idx = dict(zip(user_map['userId'].astype('int64'), user_map['user_idx'].astype('int64')))
item_to_idx = dict(zip(item_map['movieId'].astype('int64'), item_map['item_idx'].astype('int64')))
idx_to_item = dict(zip(item_map['item_idx'].astype('int64'), item_map['movieId'].astype('int64')))

ix = mf_train[['userId', 'movieId']].copy()
ix['u'] = ix['userId'].map(user_to_idx)
ix['i'] = ix['movieId'].map(item_to_idx)
ix = ix.dropna(subset=['u', 'i'])
ix['u'] = ix['u'].astype('int64')
ix['i'] = ix['i'].astype('int64')

n_users = user_map['user_idx'].max() + 1
n_items = item_map['item_idx'].max() + 1

interactions = csr_matrix(
    (np.ones(len(ix), dtype=np.float32), (ix['u'].to_numpy(), ix['i'].to_numpy())),
    shape=(n_users, n_items),
    dtype=np.float32,
)

user_item = interactions.T

val_gt_map = val_gt.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()
test_gt_map = test_gt.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()
seen_map = seen_items.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()

candidate_items = set(item_map['movieId'].astype('int64'))

# popularity fallback
pop_table_full = (
    mf_train.groupby('movieId')
    .size()
    .reset_index(name='cnt')
    .sort_values(['cnt', 'movieId'], ascending=[False, True])
)

pop_global = pop_table_full['movieId'].astype('int64').tolist()

def fallback_popularity(user_id, k):
    seen = seen_map.get(int(user_id), set())
    out = []
    for m in pop_global:
        if m not in seen:
            out.append(m)
            if len(out) >= k:
                break
    return out


def build_user_items(df, user_col='userId', item_col='movieId'):
    s = df.groupby(user_col)[item_col].apply(lambda x: list(x)).to_dict()
    return {int(k): [int(v) for v in vals] for k, vals in s.items()}


def dcg_at_k(recommended, relevant_set, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant_set:
            score += 1.0 / np.log2(i + 1)
    return score


def idcg_at_k(n_relevant, k):
    m = min(n_relevant, k)
    if m == 0:
        return 0.0
    return sum(1.0 / np.log2(i + 1) for i in range(1, m + 1))


def evaluate_ranked(pred_df, gt_map, eval_users, candidate_items, k_values):
    pred_user_items = build_user_items(pred_df.sort_values(['userId', 'rank']))

    rows = []
    for k in k_values:
        recalls, precisions, maps, ndcgs, hits = [], [], [], [], []
        rec_all = []
        lens = []

        for uid in eval_users:
            uid = int(uid)
            recs = pred_user_items.get(uid, [])[:k]
            gt = gt_map.get(uid, set())
            if isinstance(gt, list):
                gt = set(gt)

            lens.append(len(recs))
            rec_all.extend(recs)

            if len(gt) == 0:
                continue

            rel = [1 if x in gt else 0 for x in recs]
            n_hits = int(sum(rel))

            recalls.append(n_hits / len(gt))
            precisions.append(n_hits / max(k, 1))

            if n_hits == 0:
                maps.append(0.0)
            else:
                precs = []
                hit_count = 0
                for i, r in enumerate(rel, start=1):
                    if r:
                        hit_count += 1
                        precs.append(hit_count / i)
                maps.append(float(np.mean(precs)) if precs else 0.0)

            ndcgs.append(dcg_at_k(recs, gt, k) / idcg_at_k(len(gt), k))
            hits.append(1.0 if n_hits > 0 else 0.0)

        unique_recs = len(set(rec_all)) if len(rec_all) else 0
        catalog_coverage = unique_recs / max(len(candidate_items), 1)

        rows.append({
            'K': k,
            'Recall@K': float(np.mean(recalls)) if recalls else 0.0,
            'Precision@K': float(np.mean(precisions)) if precisions else 0.0,
            'MAP@K': float(np.mean(maps)) if maps else 0.0,
            'NDCG@K': float(np.mean(ndcgs)) if ndcgs else 0.0,
            'HitRate@K': float(np.mean(hits)) if hits else 0.0,
            'Catalog Coverage@K': catalog_coverage,
            'average_list_length': float(np.mean(lens)) if lens else 0.0,
            'full_topk_rate': float(np.mean([1 if l == k else 0 for l in lens])) if lens else 0.0,
            'unique_recommended_items': unique_recs,
            'candidate_items': len(candidate_items),
        })
    return pd.DataFrame(rows)


def recommend_for_user(model, user_id: int, k: int):
    u_idx = user_to_idx.get(int(user_id))
    if u_idx is None:
        return []

    user_items_mat = interactions if interactions.shape[0] == model.user_factors.shape[0] else interactions.T
    user_items_mat = user_items_mat.tocsr()[u_idx]

    ids, scores = model.recommend(
        userid=0,
        user_items=user_items_mat,
        N=k,
        filter_already_liked_items=True,
        recalculate_user=False,
    )

    out = []
    for rank, (i_idx, score) in enumerate(zip(ids, scores), start=1):
        movie_id = idx_to_item.get(int(i_idx))
        if movie_id is None:
            continue
        out.append((rank, int(movie_id), float(score)))
    return out


def build_predictions(model, eval_users_df, split_name: str, k: int):
    rows = []
    users = eval_users_df['userId'].drop_duplicates().astype('int64').tolist()
    for uid in users:
        recs = recommend_for_user(model, uid, k)
        if len(recs) == 0:
            fallback = fallback_popularity(uid, k)
            recs = [(rank, int(mid), 0.0) for rank, mid in enumerate(fallback, start=1)]
        for rank, movie_id, score in recs:
            rows.append({'userId': uid, 'rank': rank, 'movieId': movie_id, 'score': score, 'split': split_name})
    return pd.DataFrame(rows)

## 4.1 Тюнинг ALS по Recall@10

In [4]:
param_space = {
    'factors': [32, 64, 96, 128],
    'regularization': [0.005, 0.01, 0.05, 0.1],
    'iterations': [20, 40, 60],
    'alpha': [10.0, 20.0, 40.0],
}

N_RUNS = 12

rng = np.random.default_rng(RANDOM_SEED)
all_cfgs = []
for f in param_space['factors']:
    for reg in param_space['regularization']:
        for iters in param_space['iterations']:
            for alpha in param_space['alpha']:
                all_cfgs.append({'factors': f, 'regularization': reg, 'iterations': iters, 'alpha': alpha})

if N_RUNS < len(all_cfgs):
    idx = rng.choice(len(all_cfgs), size=N_RUNS, replace=False)
    cfgs = [all_cfgs[i] for i in sorted(idx)]
else:
    cfgs = all_cfgs

# cache utils
import hashlib
ALS_CACHE_DIR = EXP_DIR / 'als_cache'
ALS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ALS_RESULTS_CSV = EXP_DIR / 'als_tuning_results.csv'

if ALS_RESULTS_CSV.exists():
    _prev = pd.read_csv(ALS_RESULTS_CSV)
    done_keys = set(_prev['config_key'].astype(str).tolist())
else:
    done_keys = set()

results = []
best = None

for run_idx, cfg in enumerate(cfgs, start=1):
    cfg_key = hashlib.md5(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
    if cfg_key in done_keys:
        print(f"skip cached config {cfg_key}")
        continue

    t0 = time.time()

    model = AlternatingLeastSquares(
        factors=cfg['factors'],
        regularization=cfg['regularization'],
        iterations=cfg['iterations'],
        alpha=cfg['alpha'],
        num_threads=N_CPU_THREADS,
        random_state=RANDOM_SEED,
    )

    conf = user_item.copy().astype('float32')
    conf.data = 1.0 + float(cfg['alpha']) * conf.data

    model.fit(conf)

    val_pred = build_predictions(model, eval_users_val, 'validation', K_MAX)
    val_metrics = evaluate_ranked(val_pred, val_gt_map, eval_users_val['userId'].astype('int64').tolist(), candidate_items, K_VALUES)
    target_row = val_metrics[val_metrics['K'] == TARGET_K].iloc[0].to_dict()

    row = {
        'run_idx': run_idx,
        'config_key': cfg_key,
        'runtime_sec': time.time() - t0,
        **cfg,
        'val_recall_at_k': float(target_row['Recall@K']),
        'val_precision_at_k': float(target_row['Precision@K']),
        'val_map_at_k': float(target_row['MAP@K']),
        'val_ndcg_at_k': float(target_row['NDCG@K']),
        'val_hitrate_at_k': float(target_row['HitRate@K']),
    }

    results.append(row)

    # cache factors
    factors_path = ALS_CACHE_DIR / f"als_factors_{cfg_key}.npz"
    np.savez(factors_path, user_factors=model.user_factors, item_factors=model.item_factors)

    # checkpoint CSV
    pd.DataFrame(results).to_csv(ALS_RESULTS_CSV, index=False)

    if best is None or row['val_recall_at_k'] > best['val_recall_at_k']:
        best = row

    print(f"run {run_idx}/{len(cfgs)} | recall@{TARGET_K}={row['val_recall_at_k']:.5f}")

if len(results) == 0:
    if ALS_RESULTS_CSV.exists():
        results_df = pd.read_csv(ALS_RESULTS_CSV)
    else:
        results_df = pd.DataFrame()
else:
    results_df = pd.DataFrame(results).sort_values('val_recall_at_k', ascending=False).reset_index(drop=True)

if len(results_df) == 0:
    print('Нет новых результатов (возможно, все конфиги уже закэшированы).')
else:
    display(results_df.head(10))


skip cached config d060cd558351afc01be8426c8f0ce8ec
skip cached config 3d67bca6df41f94ba1e3857e80e34a79
skip cached config bbccd75ca273d3e488da630ed138f991
skip cached config 6de40c1af7dcfa26135b0a041b87150f
skip cached config 48296344ba291cf1a5f36afd34d0629e
skip cached config 2721f181cd8adbf1d7b90ed58e17da28
skip cached config 6e9b62e397feced3721b2a6406b6a8f3
skip cached config 5a6f8b46db4cb242f8c320e8dec6f533
skip cached config 2ac3c3b226bf86910bc47703980f7106
skip cached config f4a2ade016409952c8e4b1688c530ca4
skip cached config 34c4e43141cf9042012bf21465836f57
skip cached config abbead82d86454734ca795852d187252


,run_idx,config_key,runtime_sec,factors,regularization,iterations,alpha,val_recall_at_k,val_precision_at_k,val_map_at_k,val_ndcg_at_k,val_hitrate_at_k
0,1,d060cd558351afc01be8426c8f0ce8ec,0.516452,32,0.010,20,40.0,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,3d67bca6df41f94ba1e3857e80e34a79,0.950863,32,0.010,40,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
2,3,bbccd75ca273d3e488da630ed138f991,0.439594,32,0.100,20,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
3,4,6de40c1af7dcfa26135b0a041b87150f,1.815670,64,0.050,40,40.0,0.008754,0.011111,0.017196,0.008431,0.111111
4,5,48296344ba291cf1a5f36afd34d0629e,3.314517,96,0.005,40,10.0,0.008754,0.011111,0.111111,0.024455,0.111111
5,6,2721f181cd8adbf1d7b90ed58e17da28,4.952392,96,0.010,60,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
6,7,6e9b62e397feced3721b2a6406b6a8f3,4.731127,96,0.050,60,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
7,8,5a6f8b46db4cb242f8c320e8dec6f533,3.514182,96,0.100,40,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
8,9,2ac3c3b226bf86910bc47703980f7106,2.835721,128,0.010,20,20.0,0.000000,0.000000,0.000000,0.000000,0.000000
9,10,f4a2ade016409952c8e4b1688c530ca4,2.853059,128,0.100,20,20.0,0.000000,0.000000,0.000000,0.000000,0.000000


## 4.2 Two-stage baseline (MF+popularity → CatBoost)

In [5]:
# anti-leakage split
train_full['timestamp'] = pd.to_numeric(train_full['timestamp'], errors='coerce')
inner_q = 0.8
train_cut_ts = int(train_full['timestamp'].quantile(inner_q))

train_gen = train_full[train_full['timestamp'] <= train_cut_ts].copy()
train_rank = train_full[train_full['timestamp'] > train_cut_ts].copy()

# popularity
pop_table = (
    train_gen.groupby('movieId')
    .agg(positive_interactions=('userId', 'size'), unique_users=('userId', 'nunique'))
    .reset_index()
    .sort_values(['positive_interactions', 'unique_users', 'movieId'], ascending=[False, False, True])
)

pop_table['popularity_rank'] = np.arange(1, len(pop_table) + 1)
pop_global = pop_table['movieId'].astype('int64').tolist()

train_gen_seen = train_gen.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()
eval_seen = seen_items.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()

train_rank_users = sorted(train_rank['userId'].drop_duplicates().astype('int64').tolist())
val_users = sorted(eval_users_val['userId'].drop_duplicates().astype('int64').tolist())
test_users = sorted(eval_users_test['userId'].drop_duplicates().astype('int64').tolist())

pop_rank_map = pop_table.set_index('movieId')['popularity_rank'].to_dict()
pop_score_map = pop_table.set_index('movieId')['positive_interactions'].to_dict()
max_pop = max(pop_score_map.values()) if len(pop_score_map) else 1.0

# best MF
if best is None:
    best = results_df.iloc[0].to_dict()

mf_model = AlternatingLeastSquares(
    factors=best['factors'],
    regularization=best['regularization'],
    iterations=best['iterations'],
    alpha=best['alpha'],
    num_threads=N_CPU_THREADS,
    random_state=RANDOM_SEED,
)

conf = user_item.copy().astype('float32')
conf.data = 1.0 + float(best['alpha']) * conf.data
mf_model.fit(conf)


def top_pop_unseen(seen_set, n):
    out = []
    for m in pop_global:
        if m not in seen_set:
            out.append(m)
            if len(out) >= n:
                break
    return out


def top_mf_unseen(user_id, seen_set, n):
    u_idx = user_to_idx.get(int(user_id))
    if u_idx is None:
        return []
    user_items_mat = interactions if interactions.shape[0] == mf_model.user_factors.shape[0] else interactions.T
    user_items_mat = user_items_mat.tocsr()[u_idx]
    ids, scores = mf_model.recommend(
        userid=0,
        user_items=user_items_mat,
        N=n + len(seen_set),
        filter_already_liked_items=True,
        recalculate_user=False,
    )
    out = []
    for rank, (i_idx, score) in enumerate(zip(ids, scores), start=1):
        mid = idx_to_item.get(int(i_idx))
        if mid is None or mid in seen_set:
            continue
        out.append((int(mid), float(score), rank))
        if len(out) >= n:
            break
    return out


def build_candidates(user_ids, seen_map, mf_candidates, pop_candidates, final_pool_max):
    rows = []
    for uid in tqdm(user_ids):
        uid = int(uid)
        seen = seen_map.get(uid, set())
        pop_items = top_pop_unseen(seen, pop_candidates)
        mf_items = top_mf_unseen(uid, seen, mf_candidates)

        cand = {}
        for r, m in enumerate(pop_items, start=1):
            d = cand.setdefault(int(m), {'userId': uid, 'movieId': int(m), 'from_pop': 0, 'from_mf': 0, 'pop_rank': None, 'pop_score': None, 'mf_rank': None, 'mf_score': None})
            d['from_pop'] = 1
            d['pop_rank'] = r
            d['pop_score'] = pop_score_map.get(int(m), 0)

        for m, score, rank in mf_items:
            d = cand.setdefault(int(m), {'userId': uid, 'movieId': int(m), 'from_pop': 0, 'from_mf': 0, 'pop_rank': None, 'pop_score': None, 'mf_rank': None, 'mf_score': None})
            d['from_mf'] = 1
            d['mf_rank'] = rank
            d['mf_score'] = score

        cands = list(cand.values())
        cands.sort(key=lambda x: (
            x['mf_rank'] if x['mf_rank'] is not None else 10**9,
            x['pop_rank'] if x['pop_rank'] is not None else 10**9,
        ))
        cands = cands[:final_pool_max]

        for rank, c in enumerate(cands, start=1):
            c['candidate_rank'] = rank
            rows.append(c)

    return pd.DataFrame(rows)

MF_CANDIDATES = 600
POP_CANDIDATES = 400
FINAL_POOL_MAX = 1000

cand_train = build_candidates(train_rank_users, train_gen_seen, MF_CANDIDATES, POP_CANDIDATES, FINAL_POOL_MAX)
cand_val = build_candidates(val_users, eval_seen, MF_CANDIDATES, POP_CANDIDATES, FINAL_POOL_MAX)
cand_test = build_candidates(test_users, eval_seen, MF_CANDIDATES, POP_CANDIDATES, FINAL_POOL_MAX)

100%|██████████| 16/16 [00:00<00:00, 399.97it/s]


## 4.3 CatBoost на базовых фичах (кандидаты уже готовы)

Используем кандидатов из MF+popularity и базовые фичи из train_gen: user/item статистики,
жанры и простые генераторные сигналы.


In [6]:
# === Base feature maps (как в 03_pairwise_features_hard_negatives_mf) ===
# user stats
user_hist_len = train_gen.groupby('userId').size().to_dict()
user_mean_rating = train_gen.groupby('userId')['rating'].mean().to_dict() if 'rating' in train_gen.columns else {}

# item stats
item_support = train_gen.groupby('movieId').size().to_dict()
item_mean_rating = train_gen.groupby('movieId')['rating'].mean().to_dict() if 'rating' in train_gen.columns else {}

# metadata maps
item_n_genres = {}
item_release_year = {}
item_genres_set = {}

if item_meta is not None and 'movieId' in item_meta.columns:
    meta = item_meta.copy()

    def parse_genres(v):
        if isinstance(v, list):
            return [str(x).strip() for x in v if str(x).strip()]
        if pd.isna(v):
            return []
        s = str(v).strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
        if '|' in s:
            return [x.strip() for x in s.split('|') if x.strip() and x.strip() != '(no genres listed)']
        return []

    if 'genres_list' in meta.columns:
        meta['_genres'] = meta['genres_list'].apply(parse_genres)
    elif 'genres' in meta.columns:
        meta['_genres'] = meta['genres'].apply(parse_genres)
    else:
        meta['_genres'] = [[] for _ in range(len(meta))]

    if 'release_year' in meta.columns:
        meta['_year'] = pd.to_numeric(meta['release_year'], errors='coerce')
    else:
        meta['_year'] = np.nan

    for _, r in meta.iterrows():
        m = int(r['movieId'])
        gs = set([g for g in r['_genres'] if g])
        item_genres_set[m] = gs
        item_n_genres[m] = len(gs)
        item_release_year[m] = float(r['_year']) if pd.notna(r['_year']) else np.nan

# user genre profile
user_genre_pref = defaultdict(lambda: defaultdict(int))
if len(item_genres_set) > 0:
    for _, r in train_gen[['userId', 'movieId']].iterrows():
        u = int(r['userId'])
        m = int(r['movieId'])
        for g in item_genres_set.get(m, set()):
            user_genre_pref[u][g] += 1

def genre_overlap_ratio(user_id, movie_id):
    gs = item_genres_set.get(int(movie_id), set())
    if len(gs) == 0:
        return 0.0
    pref = user_genre_pref.get(int(user_id), {})
    s = sum(pref.get(g, 0) for g in gs)
    denom = sum(pref.values())
    return float(s / denom) if denom > 0 else 0.0

# labels for train/val/test
train_rank_gt_map = train_rank.groupby('userId')['movieId'].apply(lambda s: set(s.astype('int64'))).to_dict()

def add_label(cand_df, gt_map):
    y = []
    for uid, mid in zip(cand_df['userId'], cand_df['movieId']):
        gt = gt_map.get(int(uid), set())
        y.append(1 if int(mid) in gt else 0)
    out = cand_df.copy()
    out['target'] = y
    return out

train_ds = add_label(cand_train, train_rank_gt_map)
val_ds = add_label(cand_val, val_gt_map)
test_ds = add_label(cand_test, test_gt_map)

def enrich(df):
    out = df.copy()
    out['user_hist_len'] = out['userId'].map(user_hist_len).fillna(0)
    out['user_mean_rating'] = out['userId'].map(user_mean_rating).fillna(0)
    out['item_support'] = out['movieId'].map(item_support).fillna(0)
    out['item_mean_rating'] = out['movieId'].map(item_mean_rating).fillna(0)
    out['item_n_genres'] = out['movieId'].map(item_n_genres).fillna(0)
    out['item_release_year'] = out['movieId'].map(item_release_year).fillna(np.nan)
    out['genre_overlap'] = [genre_overlap_ratio(u, m) for u, m in zip(out['userId'], out['movieId'])]

    # generator signals
    out['pop_score_norm'] = out['pop_score'].fillna(0) / (out['pop_score'].fillna(0).max() + 1e-9)
    out['mf_score_norm'] = out['mf_score'].fillna(0) / (out['mf_score'].fillna(0).max() + 1e-9)

    return out

train_ds = enrich(train_ds)
val_ds = enrich(val_ds)
test_ds = enrich(test_ds)

# feature set
feature_cols = [
    'from_pop','from_mf','pop_rank','pop_score','mf_rank','mf_score','candidate_rank',
    'user_hist_len','user_mean_rating','item_support','item_mean_rating',
    'item_n_genres','item_release_year','genre_overlap','pop_score_norm','mf_score_norm'
]
for df in [train_ds, val_ds, test_ds]:
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0
    df[feature_cols] = df[feature_cols].fillna(0)

# CatBoost training (base features)
train_pool = Pool(data=train_ds[feature_cols], label=train_ds['target'].astype(int), group_id=train_ds['userId'].astype(int))
val_pool = Pool(data=val_ds[feature_cols], label=val_ds['target'].astype(int), group_id=val_ds['userId'].astype(int))

cb_params = {
    'loss_function': 'YetiRankPairwise',
    'eval_metric': 'NDCG:top=10',
    'depth': 8,
    'learning_rate': 0.1,
    'l2_leaf_reg': 4.0,
    'iterations': 300,
    'random_seed': RANDOM_SEED,
    'verbose': 50,
}
if USE_GPU:
    cb_params['task_type'] = 'GPU'
    cb_params['devices'] = '0'

cb_base = CatBoostRanker(**cb_params)
cb_base.fit(train_pool, eval_set=val_pool)

def score_and_rank(df, model):
    out = df.copy()
    out['rank_score'] = model.predict(out[feature_cols])
    out = out.sort_values(['userId','rank_score'], ascending=[True, False]).copy()
    out['rank'] = out.groupby('userId').cumcount() + 1
    return out

val_scored = score_and_rank(val_ds, cb_base)
test_scored = score_and_rank(test_ds, cb_base)

val_metrics = evaluate_ranked(val_scored, val_gt_map, val_users, candidate_items, K_VALUES)
test_metrics = evaluate_ranked(test_scored, test_gt_map, test_users, candidate_items, K_VALUES)

display(val_metrics)
display(test_metrics)

# save artifacts
cb_model_path = MODELS_DIR / 'catboost_base.cbm'
cb_base.save_model(cb_model_path)

val_metrics_path = METRICS_DIR / 'catboost_base_val_metrics.csv'
test_metrics_path = METRICS_DIR / 'catboost_base_test_metrics.csv'
val_metrics.to_csv(val_metrics_path, index=False)
test_metrics.to_csv(test_metrics_path, index=False)

val_pred_path = PRED_DIR / 'catboost_base_val_predictions.csv'
test_pred_path = PRED_DIR / 'catboost_base_test_predictions.csv'
val_scored[['userId','movieId','rank','rank_score']].to_csv(val_pred_path, index=False)
test_scored[['userId','movieId','rank','rank_score']].to_csv(test_pred_path, index=False)

print('Saved:', cb_model_path)
print('Saved:', val_metrics_path)
print('Saved:', test_metrics_path)


0:	test: 0.2222222	best: 0.2222222 (0)	total: 1.1s	remaining: 5m 29s
50:	test: 0.2969358	best: 0.3066361 (46)	total: 43.4s	remaining: 3m 31s
100:	test: 0.3074637	best: 0.3103132 (72)	total: 1m 15s	remaining: 2m 28s
150:	test: 0.3063794	best: 0.3103132 (72)	total: 1m 50s	remaining: 1m 49s
200:	test: 0.3033232	best: 0.3115991 (158)	total: 2m 26s	remaining: 1m 12s
250:	test: 0.3045856	best: 0.3141908 (243)	total: 3m 1s	remaining: 35.4s
299:	test: 0.3061865	best: 0.3141908 (243)	total: 3m 32s	remaining: 0us

bestTest = 0.3141908085
bestIteration = 243

Shrink model to first 244 iterations.


,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,5,0.020858,0.055556,0.177778,0.068750,0.277778,0.007463,5.0,1.0,26,3484
1,10,0.023222,0.038889,0.170106,0.054371,0.277778,0.012342,10.0,1.0,43,3484
2,20,0.043284,0.030556,0.127799,0.055120,0.277778,0.023823,20.0,1.0,83,3484
3,50,0.097447,0.024444,0.117829,0.072937,0.388889,0.050804,50.0,1.0,177,3484


,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,5,0.024069,0.08750,0.119792,0.098001,0.1250,0.008037,5.0,1.0,28,3484
1,10,0.067930,0.10000,0.120939,0.106706,0.2500,0.012916,10.0,1.0,45,3484
2,20,0.106294,0.08750,0.122927,0.119395,0.3125,0.024110,20.0,1.0,84,3484
3,50,0.133925,0.05125,0.114916,0.112007,0.4375,0.050804,50.0,1.0,177,3484


Saved: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_small/models/two_stage_mf_catboost/catboost_base.cbm
Saved: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_small/metrics/two_stage_mf_catboost/catboost_base_val_metrics.csv
Saved: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_small/metrics/two_stage_mf_catboost/catboost_base_test_metrics.csv


## 4.4 Pairwise + hard/random negatives → CatBoost и сравнение с базовым

Добавляем pairwise-фичи и формируем hard/random negatives для усиления ранкера.


In [7]:
# === Pairwise features ===
for df in [train_ds, val_ds, test_ds]:
    df['user_item_rating_gap'] = df['user_mean_rating'] - df['item_mean_rating']
    df['user_hist_len_log'] = np.log1p(df['user_hist_len'])
    df['item_support_log'] = np.log1p(df['item_support'])
    df['pop_rank_inv'] = 1.0 / (df['pop_rank'].fillna(1e9))
    df['mf_rank_inv'] = 1.0 / (df['mf_rank'].fillna(1e9))

pairwise_cols = ['user_item_rating_gap','user_hist_len_log','item_support_log','pop_rank_inv','mf_rank_inv']
feature_cols_hard = feature_cols + pairwise_cols
for df in [train_ds, val_ds, test_ds]:
    for c in feature_cols_hard:
        if c not in df.columns:
            df[c] = 0
    df[feature_cols_hard] = df[feature_cols_hard].replace([np.inf, -np.inf], 0).fillna(0)
for df in [train_ds, val_ds, test_ds]:
    df[feature_cols_hard] = df[feature_cols_hard].apply(pd.to_numeric, errors='coerce')
    df[feature_cols_hard] = df[feature_cols_hard].replace([np.inf, -np.inf], 0).fillna(0).astype('float32')

# === Hard negatives + random negatives ===
HARD_NEG_RATIO = 3
RAND_NEG_RATIO = 1
MAX_NEG_PER_USER = 200

hard_neg = train_ds[train_ds['target'] == 0].copy()
hard_neg = hard_neg.sort_values(['userId','candidate_rank']).copy()
hard_rows = []
for uid, group in hard_neg.groupby('userId'):
    pos_cnt = int(train_ds[(train_ds['userId'] == uid) & (train_ds['target'] == 1)].shape[0])
    if pos_cnt == 0:
        continue
    take = min(MAX_NEG_PER_USER, pos_cnt * HARD_NEG_RATIO)
    hard_rows.append(group.head(take))
hard_neg = pd.concat(hard_rows, ignore_index=True) if hard_rows else hard_neg.head(0)

rand_neg = train_ds[train_ds['target'] == 0].copy()
rand_rows = []
for uid, group in rand_neg.groupby('userId'):
    pos_cnt = int(train_ds[(train_ds['userId'] == uid) & (train_ds['target'] == 1)].shape[0])
    if pos_cnt == 0:
        continue
    take = min(MAX_NEG_PER_USER, pos_cnt * RAND_NEG_RATIO)
    if len(group) <= take:
        rand_rows.append(group)
    else:
        rand_rows.append(group.sample(take, random_state=RANDOM_SEED))
rand_neg = pd.concat(rand_rows, ignore_index=True) if rand_rows else rand_neg.head(0)

pos_train = train_ds[train_ds['target'] == 1].copy()
train_final = pd.concat([pos_train, hard_neg, rand_neg], ignore_index=True)
train_final = train_final.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print('train_final shape:', train_final.shape)
print('positives:', train_final['target'].sum())

# ensure group_id contiguous for CatBoost
train_final = train_final.sort_values(['userId','candidate_rank','movieId']).reset_index(drop=True)
val_ds = val_ds.sort_values(['userId','candidate_rank','movieId']).reset_index(drop=True)

# === CatBoost on hardneg ===
train_pool_h = Pool(data=train_final[feature_cols_hard], label=train_final['target'].astype(int), group_id=train_final['userId'].astype(int))
val_pool_h = Pool(data=val_ds[feature_cols_hard], label=val_ds['target'].astype(int), group_id=val_ds['userId'].astype(int))

cb_params_h = cb_params.copy()
cb_params_h['iterations'] = 500
cb_hard = CatBoostRanker(**cb_params_h)
cb_hard.fit(train_pool_h, eval_set=val_pool_h)

def score_and_rank_h(df, model):
    out = df.copy()
    out['rank_score'] = model.predict(out[feature_cols_hard])
    out = out.sort_values(['userId','rank_score'], ascending=[True, False]).copy()
    out['rank'] = out.groupby('userId').cumcount() + 1
    return out

val_scored_h = score_and_rank_h(val_ds, cb_hard)
test_scored_h = score_and_rank_h(test_ds, cb_hard)

val_metrics_h = evaluate_ranked(val_scored_h, val_gt_map, val_users, candidate_items, K_VALUES)
test_metrics_h = evaluate_ranked(test_scored_h, test_gt_map, test_users, candidate_items, K_VALUES)

display(val_metrics_h)
display(test_metrics_h)

# save artifacts
cb_hard_path = MODELS_DIR / 'catboost_hardneg.cbm'
cb_hard.save_model(cb_hard_path)

val_metrics_h_path = METRICS_DIR / 'catboost_hardneg_val_metrics.csv'
test_metrics_h_path = METRICS_DIR / 'catboost_hardneg_test_metrics.csv'
val_metrics_h.to_csv(val_metrics_h_path, index=False)
test_metrics_h.to_csv(test_metrics_h_path, index=False)

val_pred_h_path = PRED_DIR / 'catboost_hardneg_val_predictions.csv'
test_pred_h_path = PRED_DIR / 'catboost_hardneg_test_predictions.csv'
val_scored_h[['userId','movieId','rank','rank_score']].to_csv(val_pred_h_path, index=False)
test_scored_h[['userId','movieId','rank','rank_score']].to_csv(test_pred_h_path, index=False)

# comparison at TARGET_K
base_k_val = val_metrics[val_metrics['K'] == TARGET_K].iloc[0].to_dict()
base_k_test = test_metrics[test_metrics['K'] == TARGET_K].iloc[0].to_dict()
hard_k_val = val_metrics_h[val_metrics_h['K'] == TARGET_K].iloc[0].to_dict()
hard_k_test = test_metrics_h[test_metrics_h['K'] == TARGET_K].iloc[0].to_dict()

comparison = pd.DataFrame([
    {'model': 'catboost_base', 'split': 'val', 'Recall@K': base_k_val['Recall@K'], 'Precision@K': base_k_val['Precision@K'], 'MAP@K': base_k_val['MAP@K'], 'NDCG@K': base_k_val['NDCG@K']},
    {'model': 'catboost_hardneg', 'split': 'val', 'Recall@K': hard_k_val['Recall@K'], 'Precision@K': hard_k_val['Precision@K'], 'MAP@K': hard_k_val['MAP@K'], 'NDCG@K': hard_k_val['NDCG@K']},
    {'model': 'catboost_base', 'split': 'test', 'Recall@K': base_k_test['Recall@K'], 'Precision@K': base_k_test['Precision@K'], 'MAP@K': base_k_test['MAP@K'], 'NDCG@K': base_k_test['NDCG@K']},
    {'model': 'catboost_hardneg', 'split': 'test', 'Recall@K': hard_k_test['Recall@K'], 'Precision@K': hard_k_test['Precision@K'], 'MAP@K': hard_k_test['MAP@K'], 'NDCG@K': hard_k_test['NDCG@K']},
])
display(comparison)

comparison_path = METRICS_DIR / 'catboost_base_vs_hardneg.csv'
comparison.to_csv(comparison_path, index=False)
print('Saved:', cb_hard_path)
print('Saved:', comparison_path)


train_final shape: (15678, 24)
positives: 3365
0:	test: 0.2222222	best: 0.2222222 (0)	total: 725ms	remaining: 6m 1s
50:	test: 0.2531528	best: 0.2673287 (26)	total: 20.3s	remaining: 2m 59s
100:	test: 0.2687010	best: 0.2785476 (99)	total: 47.3s	remaining: 3m 6s
150:	test: 0.2809289	best: 0.2853888 (135)	total: 1m 21s	remaining: 3m 7s
200:	test: 0.2867185	best: 0.2867185 (199)	total: 2m 4s	remaining: 3m 4s
250:	test: 0.2845435	best: 0.2867185 (199)	total: 2m 43s	remaining: 2m 41s
300:	test: 0.2920287	best: 0.2943901 (273)	total: 3m 23s	remaining: 2m 14s
350:	test: 0.2845435	best: 0.2943901 (273)	total: 4m 12s	remaining: 1m 47s
400:	test: 0.2845435	best: 0.2943901 (273)	total: 4m 55s	remaining: 1m 12s
450:	test: 0.2861165	best: 0.2943901 (273)	total: 5m 43s	remaining: 37.3s
499:	test: 0.2861165	best: 0.2943901 (273)	total: 6m 27s	remaining: 0us

bestTest = 0.2943901455
bestIteration = 273

Shrink model to first 274 iterations.


,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,5,0.025512,0.066667,0.126852,0.066283,0.222222,0.006028,5.0,1.0,21,3484
1,10,0.031685,0.038889,0.123148,0.048822,0.222222,0.012055,10.0,1.0,42,3484
2,20,0.040839,0.030556,0.093483,0.048657,0.277778,0.022675,20.0,1.0,79,3484
3,50,0.099246,0.025556,0.071811,0.067773,0.388889,0.047359,50.0,1.0,165,3484


,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,5,0.018285,0.08750,0.100521,0.078360,0.1875,0.006889,5.0,1.0,24,3484
1,10,0.020368,0.05625,0.099687,0.059166,0.1875,0.012629,10.0,1.0,44,3484
2,20,0.059486,0.05000,0.094449,0.069377,0.3125,0.023536,20.0,1.0,82,3484
3,50,0.093478,0.04250,0.079436,0.075451,0.4375,0.047646,50.0,1.0,166,3484


,model,split,Recall@K,Precision@K,MAP@K,NDCG@K
0,catboost_base,val,0.023222,0.038889,0.170106,0.054371
1,catboost_hardneg,val,0.031685,0.038889,0.123148,0.048822
2,catboost_base,test,0.067930,0.100000,0.120939,0.106706
3,catboost_hardneg,test,0.020368,0.056250,0.099687,0.059166


Saved: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_small/models/two_stage_mf_catboost/catboost_hardneg.cbm
Saved: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts_small/metrics/two_stage_mf_catboost/catboost_base_vs_hardneg.csv


## 4.5 Тюнинг CatBoost (на hardneg + pairwise)

Подбираем гиперпараметры CatBoost на train_final и оцениваем на validation.


In [9]:
from itertools import product
import hashlib

CB_GRID = {
    'depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1],
    'l2_leaf_reg': [2.0, 4.0, 8.0],
    'iterations': [300, 500],
}
MAX_CB_RUNS = 4

CB_TUNE_DIR = EXP_DIR / 'catboost_tuning'
CB_TUNE_DIR.mkdir(parents=True, exist_ok=True)
CB_RESULTS_CSV = CB_TUNE_DIR / 'catboost_tuning_results.csv'

if CB_RESULTS_CSV.exists():
    _prev = pd.read_csv(CB_RESULTS_CSV)
    done_keys = set(_prev['config_key'].astype(str).tolist())
else:
    done_keys = set()

cfgs = []
for d, lr, l2, iters in product(CB_GRID['depth'], CB_GRID['learning_rate'], CB_GRID['l2_leaf_reg'], CB_GRID['iterations']):
    cfgs.append({'depth': d, 'learning_rate': lr, 'l2_leaf_reg': l2, 'iterations': iters})

rng = np.random.default_rng(RANDOM_SEED + 7)
if MAX_CB_RUNS < len(cfgs):
    idx = rng.choice(len(cfgs), size=MAX_CB_RUNS, replace=False)
    cfgs = [cfgs[i] for i in sorted(idx)]

cb_rows = []
best_cb = None

for run_idx, cfg in enumerate(cfgs, start=1):
    cfg_key = hashlib.md5(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
    if cfg_key in done_keys:
        print(f'skip cached cb config {cfg_key}')
        continue

    params = {
        'loss_function': 'YetiRankPairwise',
        'eval_metric': 'NDCG:top=10',
        'random_seed': RANDOM_SEED,
        'verbose': False,
        **cfg
    }
    if USE_GPU:
        params['task_type'] = 'GPU'
        params['devices'] = '0'

    t0 = time.time()
    cb = CatBoostRanker(**params)
    cb.fit(train_pool_h, eval_set=val_pool_h)

    val_scored_t = score_and_rank_h(val_ds, cb)
    val_metrics_t = evaluate_ranked(val_scored_t, val_gt_map, val_users, candidate_items, K_VALUES)
    target_row = val_metrics_t[val_metrics_t['K'] == TARGET_K].iloc[0].to_dict()

    row = {
        'run_idx': run_idx,
        'config_key': cfg_key,
        'runtime_sec': time.time() - t0,
        **cfg,
        'val_recall_at_k': float(target_row['Recall@K']),
        'val_precision_at_k': float(target_row['Precision@K']),
        'val_map_at_k': float(target_row['MAP@K']),
        'val_ndcg_at_k': float(target_row['NDCG@K']),
        'val_hitrate_at_k': float(target_row['HitRate@K']),
    }
    cb_rows.append(row)

    pd.DataFrame(cb_rows).to_csv(CB_RESULTS_CSV, index=False)

    if best_cb is None or row['val_ndcg_at_k'] > best_cb['val_ndcg_at_k']:
        best_cb = row

    print('run {}/{} | val_recall@{}={:.5f}'.format(run_idx, len(cfgs), TARGET_K, row['val_recall_at_k']))

if len(cb_rows) == 0 and CB_RESULTS_CSV.exists():
    cb_results_df = pd.read_csv(CB_RESULTS_CSV)
else:
    cb_results_df = pd.DataFrame(cb_rows)

if len(cb_results_df) == 0:
    print('Нет новых результатов CatBoost (возможно, все конфиги уже закэшированы).')
else:
    cb_results_df = cb_results_df.sort_values('val_ndcg_at_k', ascending=False).reset_index(drop=True)
    display(cb_results_df.head(10))

top10_path = CB_TUNE_DIR / 'catboost_top10.csv'
if len(cb_results_df) > 0:
    cb_results_df.head(10).to_csv(top10_path, index=False)
    print('Saved:', top10_path)


skip cached cb config 5181b568181f3c1f7af0e1e616ea5134
skip cached cb config 7041d5951b8d46da73f089d2d9446382
skip cached cb config e85d170a0bae07b3f527ce3c6d6b7cf9
